In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Fast read
df = pd.read_csv("Database/master.csv", low_memory=False)

# ---------- Add arrayMic rows only if missing ----------
fp = df["file_path"].astype(str)

has_array = fp.str.contains("wav_arrayMic", regex=False).any()

if not has_array:
    head_mask = fp.str.contains("wav_headMic", regex=False)

    df_array = df.loc[head_mask].copy()
    df_array["file_path"] = df_array["file_path"].str.replace(
        "wav_headMic",
        "wav_arrayMic",
        regex=False
    )

    df = pd.concat([df, df_array], ignore_index=True)

# ---------- Severity labels ----------
severity_map = {
    "FC01": 0, "FC02": 0, "FC03": 0,
    "MC01": 0, "MC02": 0, "MC03": 0, "MC04": 0,

    "F04": 1, "M03": 1,

    "F01": 2, "F03": 2, "M01": 2,
    "M02": 2, "M04": 2, "M05": 2
}

df["severity_label"] = df["speaker_id"].map(severity_map).astype("int8")

label_name = {0: "Healthy", 1: "Mild", 2: "Severe"}
df["severity"] = df["severity_label"].map(label_name)

# ---------- Stratified split ----------
idx = df.index

train_idx, temp_idx = train_test_split(
    idx,
    test_size=0.30,
    stratify=df["severity_label"],
    random_state=42
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=df.loc[temp_idx, "severity_label"],
    random_state=42
)

# ---------- Assign split fast ----------
df["split"] = ""

df.loc[train_idx, "split"] = "train"
df.loc[val_idx, "split"] = "val"
df.loc[test_idx, "split"] = "test"

# ---------- Print stats ----------
print(df.groupby(["split", "severity_label"]).size())
print(df["split"].value_counts())

# ---------- Save once only ----------
df.to_csv("Database/master_severity_split.csv", index=False)

print("Done.")

split  severity_label
test   0                 2
       1                 1
       2                 1
train  0                 4
       1                 1
       2                 4
val    0                 1
       2                 1
Name: speaker_id, dtype: int64
split
train    10020
test      3377
val       2587
Name: count, dtype: int64
